In [1]:
# Cell 1
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from elemental_society_analysis.geochem.correctors import CLRCorrector
from elemental_society_analysis.geochem.dataset import GeochemicalDataset, CorrectionService
from elemental_society_analysis.models.detectors import AnomalyDetectorConfig, create_detector

# Cell 2 – load data
df = pd.read_csv("../data/sample/sample_geochem.csv")
df.head()

# Cell 3 – CLR enrichment
corrector = CLRCorrector()
results = corrector.apply_with_enrichment(df[df.select_dtypes("number").columns], tolerance=0.05)
results["summary"]["most_enriched_overall"]

# Cell 4 – anomaly detection
cfg = AnomalyDetectorConfig(contamination=0.08)
detector = create_detector(cfg)
dataset = GeochemicalDataset(df, df.select_dtypes("number").columns.tolist())
service = CorrectionService()
transformed = service.correct(dataset, "CLR")
detector.fit(transformed)
labels = detector.predict(transformed)
print(f"Anomalies: {(labels == -1).sum()}")

ModuleNotFoundError: No module named 'elemental_society_analysis.geochem.correctors'

In [2]:
# Cell 1 – train detector and log to MLflow
import mlflow
from elemental_society_analysis.core.analyzer import TrainingCoordinator, GeochemAnalyzer
from elemental_society_analysis.models.detectors import AnomalyDetectorConfig, create_detector
from elemental_society_analysis.geochem.dataset import CorrectionService

analyzer = GeochemAnalyzer(
    correction_service=CorrectionService(),
    transform_method="CLR",
    detector=create_detector(AnomalyDetectorConfig(contamination=0.08)),
)
coordinator = TrainingCoordinator(analyzer)

with mlflow.start_run(run_name="baseline-if"):
    metrics, result = coordinator.run_training(df, numeric_cols)
    mlflow.log_metrics({k: float(v) for k, v in metrics.items()})

ModuleNotFoundError: No module named 'elemental_society_analysis.core.analyzer'